# Description
Calculation of the number of passes using real breadcrumbs. Finished on May 18th, 2026. It is the first version it can count by boundary cell. The grid is done by rectangles

In [1]:
%load_ext autoreload
%autoreload 2
%reset -f
%matplotlib widget

In [2]:
from locallib.picarrodb import *
from locallib.query import *
from locallib.box import *
from locallib.query import *

from NOP import *

import matplotlib.pyplot as plt
from shapely import wkt


EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully


/home/sandbox/personal-repos/MutiplePassDetection/UniqueCell/r3/NOP.py:17: UserWarning: registration of accessor <class 'NOP.NOPAccessor'> under name 'nop' for type <class 'pandas.core.frame.DataFrame'> is overriding a preexisting attribute with the same name.
  @pd.api.extensions.register_dataframe_accessor("nop")


# Configuration

# Query the surveys

In [3]:
a = get_reports('Cadent',years = [2026]).execute([EU2_Conn])
report_bc = a.iloc[[700]].copy()
#report_bc.to_csv('Reports_used.csv', mode='a', header=False, index=False)
report_bc.db.set_query(query_Segments_byReport(report_table = '#TempReport'))
segments = report_bc.db.execute([EU2_Conn], source_col = 'ReportId', temp_table_name = '#TempReport')
# Create a GeoDataFrame for the segments with geometry from the Breadcrumb column
segments['geometry'] = segments['Breadcrumb'].apply(wkt.loads)
segments_gdf = gpd.GeoDataFrame(segments, geometry='geometry', crs='EPSG:4326')


In [4]:
# Get the unique SurveyIds from the segments DataFrame
survey_ids = pd.DataFrame(segments['SurveyId'].unique(), columns=['SurveyId'])
survey_ids.db.set_query(f"SELECT SA.SurveyId, Shape.STAsText() as SurveyAreaBoundary FROM SurveyArea SA WHERE SA.SurveyId IN (SELECT SurveyId FROM #TempSurveys)")
s = survey_ids.db.execute(EU2_Conn, source_col = 'SurveyId', temp_table_name = '#TempSurveys')

In [5]:
s['geometry'] = s['SurveyAreaBoundary'].apply(wkt.loads)
s_gdf = gpd.GeoDataFrame(s, geometry='geometry', crs='EPSG:4326')

In [6]:
# Get the segments for each survey (as a list of DataFrames)
segments_per_survey = [df for _, df in segments.groupby('SurveyId')]
output = []
surveys = []

survey_gdf = segments_per_survey[0].copy()
survey_gdf['geometry'] = survey_gdf['Breadcrumb'].apply(wkt.loads)
survey_gdf = gpd.GeoDataFrame(survey_gdf, geometry='geometry', crs='EPSG:4326')
utm_crs = survey_gdf.estimate_utm_crs()
survey_gdf = survey_gdf.to_crs(utm_crs)
survey_gdf.set_geometry('geometry', inplace=True)
surveys.append(survey_gdf)
survey_gdf.nop.prepare_survey_gdf()
cell = survey_gdf.nop.generate_grid()
net = survey_gdf.nop.create_network()

Center of the bounds: POINT (656870.83446 5938950.94847)
Processing angle:  0
Processing angle:  -10.0
Processing angle:  -20.0
Processing angle:  -30.0
Processing angle:  -40.0
Processing angle:  -50.0
Processing angle:  -60.0
Processing angle:  -70.0
Processing angle:  -80.0
Processing angle:  -90.0
Processing angle:  -100.0
Processing angle:  -110.0
Processing angle:  -120.0
Processing angle:  -130.0
Processing angle:  -140.0
Processing angle:  -150.0
Processing angle:  -160.0
Processing angle:  -170.0
Processing angle:  -180.0
Total boundaries: 2420
Boundaries with no NaN 'nop': 2386
Boundaries after length and spread filters: 2090
Boundaries after removing overlaps: 1598
Boundaries after removing close together cells: 1556
1556 splitters -> 1537 cells
Created network with 1537 nodes and 1557 edges.


In [7]:
s_gdf = survey_gdf.nop.survey_gdf
cell_gdf = survey_gdf.nop.cell_gdf